In [31]:
import os
# change working directory to project root
os.chdir("/Users/yensydney/Desktop/pstat197/module-2-claims-data-table-10")
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization, Dense, Dropout
from tensorflow.keras import Sequential

In [47]:
data_dir = "data"
rdata_files = ["claims-clean-example.RData", "claims-clean.RData", "claims-raw.RData"]

# pick the first file that exists
selected_file = None
for f in rdata_files:
    if os.path.exists(os.path.join(data_dir, f)):
        selected_file = f
        break

if selected_file is None:
    raise FileNotFoundError("No RData files found in data folder")

selected_path = os.path.join(data_dir, selected_file)
print("Loading:", selected_path)

rdata = pyreadr.read_r(selected_path)
print("Objects inside:", list(rdata.keys()))

# pick first object
df = rdata[next(iter(rdata.keys()))]
print(df.head())
print(df.columns)

Loading: data/claims-clean-example.RData
Objects inside: ['claims_clean']
   neo_search_transaction_id  neo_search_subject_id  \
0                 12395162.0             11497914.0   
1                 12394582.0             11499214.0   
2                 12384260.0             11485861.0   
3                 12363093.0             11481705.0   
4                 12376392.0             11488536.0   

                                        original_url  \
0  http://hosting-22647.tributes.com/obituary/sho...   
1  https://www.localcrimenews.com/welcome/arrest/...   
2  https://www.bustedmugshots.com/florida/miami-d...   
3  https://www.policearrests.com/florida-arrest-r...   
4  https://www.bustedmugshots.com/tennessee/memph...   

                                            text_tmp  \
0  <!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 S...   
1  <!DOCTYPE html>\n<html lang="en">\n<head>\n   ...   
2  <!DOCTYPE html>\n<html>\n<head>\n<meta charset...   
3  <!DOCTYPE HTML>\n<html lang="en

In [48]:
# automatically find columns
text_col = next((c for c in df.columns if "text_clean" in c.lower()), None)
label_col = next((c for c in df.columns if "bclass" in c.lower()), None)
id_col = next((c for c in df.columns if c == ".id"), None)

print("Text column:", text_col)
print("Label column:", label_col)
print("ID column:", id_col)

# convert to proper formats
texts = df[text_col].astype(str).tolist()
labels = df[label_col].astype("category").cat.codes.values  # convert to 0/1
ids = df[id_col].tolist() if id_col else list(range(len(texts)))

print("Number of examples:", len(texts))


Text column: text_clean
Label column: bclass
ID column: .id
Number of examples: 2140


In [49]:
X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
    texts, labels, ids, test_size=0.2, random_state=110122, stratify=labels
)

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))


Training samples: 1712
Test samples: 428


In [53]:
vectorizer = TextVectorization(
    standardize=None,
    split="whitespace",
    output_mode="tf_idf",
    max_tokens=None,
    ngrams=None,
    ragged=False,
    # set input shape for sequential
    input_shape=(1,)  
)
vectorizer.adapt(X_train)


In [54]:
model = Sequential([
    vectorizer,
    Dropout(0.2),
    Dense(25, activation="relu"),
    Dropout(0.2),
    Dense(1, activation="sigmoid")
])

model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["binary_accuracy"]
)

# no need to call model(X_train[:1])
model.summary()


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization_5            │ (None, 36001)          │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 36001)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 25)             │       900,050 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 25)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │            26 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 900,076 (3.43 MB)

 Trainable params: 900,076 (3.43 MB)

 Non-trainable params: 0 (0.00 B)

In [58]:
from tensorflow.keras.layers import TextVectorization
import numpy as np

# 1️⃣ Split your data (you already did this)
X_train_split, X_val, y_train_split, y_val = train_test_split(
    X_train, y_train, test_size=0.3, random_state=42
)

# 2️⃣ Vectorize your text
max_tokens = 10000   # max words in vocabulary
max_len = 100        # max length of sequence

vectorize_layer = TextVectorization(
    max_tokens=max_tokens,
    output_mode='int',
    output_sequence_length=max_len
)

# Fit vectorizer on training data
vectorize_layer.adapt(np.array(X_train_split))

# Convert text to numeric sequences
X_train_seq = vectorize_layer(np.array(X_train_split))
X_val_seq = vectorize_layer(np.array(X_val))

# 3️⃣ Convert labels to numpy arrays (if they aren't already)
y_train_arr = np.array(y_train_split)
y_val_arr = np.array(y_val)

# 4️⃣ Train the model
history = model.fit(
    X_train_seq,
    y_train_arr,
    validation_data=(X_val_seq, y_val_arr),
    epochs=5,
    batch_size=32
)


Epoch 1/5


ValueError: Exception encountered when calling TextVectorization.call().

[1mWhen using `TextVectorization` to tokenize strings, the input rank must be 1 or the last shape dimension must be 1. Received: inputs.shape=(None, 100) with rank=2[0m

Arguments received by TextVectorization.call():
  • inputs=tf.Tensor(shape=(None, 100), dtype=float32)